# CISC 886 — Section 5: Model Fine-Tuning (FINAL CORRECTED)

**Project**: E-commerce Business Intelligence Chatbot  
**Model**: TinyLlama-1.1B-Chat  
**Technique**: QLoRA (Quantized Low-Rank Adaptation)  
**Library**: Unsloth + HuggingFace TRL  
**Hardware**: Google Colab T4 GPU (16 GB VRAM)  

> ⚠️ **REQUIRED**: Set runtime to **T4 GPU** via Runtime → Change runtime type

## Step 1 — Install Dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q boto3
print('✅ Dependencies installed')

## Step 2 — S3 Login & Download Data

In [ ]:
import boto3, os
from getpass import getpass

aws_key    = getpass('AWS Access Key ID: ')
aws_secret = getpass('AWS Secret Access Key: ')

s3 = boto3.client('s3', aws_access_key_id=aws_key, aws_secret_access_key=aws_secret, region_name='us-east-1')
BUCKET = '25fltp-ecom-chatbot'

os.makedirs('./data', exist_ok=True)
print('Downloading data...')
s3.download_file(BUCKET, 'processed/train.jsonl/part-00000', './data/train.jsonl')
print('✅ Downloaded train.jsonl')

## Step 3 — Load Model & Base Test

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/tinyllama-chat-bnb-4bit',
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

PROMPT_1 = 'CEO REQUEST: Is our Electronics division performing well and who are our main competitor threats?'
FastLanguageModel.for_inference(model)
inputs = tokenizer([f'### Instruction:\n{PROMPT_1}\n\n### Response:\n'], return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=150, temperature=0.7, use_cache=True)
base_resp = tokenizer.batch_decode(out)[0].split('### Response:')[-1].strip()
print('=== BASE RESPONSE ===')
print(base_resp)

## Step 4 — Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 16, lora_dropout = 0, bias = 'none',
    use_gradient_checkpointing = 'unsloth', random_state = 42,
)
print('✅ LoRA applied')

## Step 5 — Dataset & Formatting Function (FIXED)

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files='./data/train.jsonl', split='train')

# This function solves the 'formatting_func' RuntimeError
def formatting_prompts_func(examples):
    texts = examples['text']
    return { 'text' : texts }

print('✅ Dataset loaded & formatting function defined')

## Step 6 — Hyperparameter Table

In [ ]:
print('=== HYPERPARAMETER TABLE ===')
print(f'{"Parameter":<30} {"Value"}')
print('-'*45)
print(f'{"Learning Rate":<30} 2e-4')
print(f'{"Batch Size":<30} 2')
print(f'{"Epochs":<30} 3')
print(f'{"LoRA Rank (r)":<30} 16')
print(f'{"Quantization":<30} 4-bit (bitsandbytes)')

## Step 7 — Train Model (With Corrected Parameters)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = 'text', # Specify this OR formatting_func
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = 'adamw_8bit',
        output_dir = './outputs',
    ),
)

trainer.train()
print('✅ Training complete')

## Step 8 — Export & Upload to S3

In [ ]:
import glob
model.save_pretrained_gguf('model', tokenizer, quantization_method = 'q4_k_m')

gguf_path = glob.glob('model/*.gguf')[0]
print(f'Uploading {gguf_path} to S3...')
s3.upload_file(gguf_path, BUCKET, 'model/tinyllama-chat.Q4_K_M.gguf')
print('✅ Done! Reload your EC2 Chatbot.')